# 02 · BM25 and entity recognition

Fix the gap from notebook 01 with keyword search, and see what keyword search still cannot do.

| | |
| --- | --- |
| **Needs** | `00_setup.ipynb` in the same folder. The first code cell runs it for you. |
| **LLM** | Optional. With `RUN_LLM = False` (the default in `00_setup.ipynb`) you still see everything retrieval does. |
| **Previous / next** | [`01_vector_search.ipynb`](01_vector_search.ipynb) / [`03_hybrid_search.ipynb`](03_hybrid_search.ipynb) |

*The code mirrors the [`rag-retrieval` journey](https://github.com/instaclustr/labs/tree/main/rag-retrieval) in `instaclustr/labs`. Lines marked "Mirrors" name the file in the journey that each piece comes from. This notebook only searches: it never writes to the cluster.*

### Run the setup

The next cell runs `00_setup.ipynb`. It connects to the cluster, loads the two small models, and defines the helper functions used below. You should see `[PASS]` lines. If you see `[FAIL]`, fix that before going on, using the notes in `00_setup.ipynb`.

In [ ]:
%run ./00_setup.ipynb

## Baseline BM25 RAG

**The fix.** BM25 is keyword search: it scores recipes by how many of the question's words they contain, weighted by how rare each word is. A rare word, such as a dish name, pulls the recipes that contain it to the top of the list.

**A small boost from entities.** spaCy tags the entities in the question (here, the dish name), and recipes whose own entity list contains one get an extra boost.

**What it still costs.** In the journey, this step needs a separate NER service running next to everything else. Notebook 03 removes that.

*Mirrors:* `scripts/baseline-bm25-rag/src/query.py` and `build_recipe_bm25_query()` in `support/common/opensearch_client.py`.

In [ ]:
BM25_SYSTEM = "Answer using ONLY the provided context below. If the context lacks the answer, say so plainly."


def bm25_query(text, k, entities=None, tier=None, exclude=None, require=None, recipe_ids=None):
    """BM25 over name and text (`source` and `entities` are exact-match fields), with optional hard filters and an entity boost."""
    filters, must_not, should = [], [], []
    if recipe_ids:
        filters.append({"terms": {"recipe_id": list(recipe_ids)}})
    if tier:
        filters.append({"term": {"tier": tier}})
    for field, values in (exclude or {}).items():
        must_not.append({"terms": {field: [v.lower() for v in values]}})
    for field, values in (require or {}).items():
        filters += [{"term": {field: v.lower()}} for v in values]
    if entities:
        should.append({"constant_score": {"filter": {"terms": {"entities": entities}}, "boost": 6.0}})
    must = [{"multi_match": {"query": text, "fields": ["recipe_name^4", "entities^3", "text^2", "source"],
                             "type": "best_fields", "operator": "or"}}]
    bool_query = {"must": must, "filter": filters, "must_not": must_not}
    if should:
        bool_query["should"] = should
    return {"size": k, "_source": RECIPE_FIELDS + ["entities"], "query": {"bool": bool_query}}


def bm25_search(question, k=10, index=None, **kw):
    entities = extract_entities(question)
    return normalize_hits(search(index or IDX["bm25_baseline"], bm25_query(question, k, entities=entities, **kw))), entities


def ask_bm25(question, top_k=10):
    hits, entities = bm25_search(question, top_k)
    prompt = f"Context:\n{build_context(hits, max_chars=1200)}\n\nQuestion: {question}"
    return answer(prompt, BM25_SYSTEM), hits, entities


for question, domain in [(Q_GRATEFUL, "thegratefulgirlcooks"), (Q_WOMENS, "womensweeklyfood")]:
    reply, hits, entities = ask_bm25(question)
    print("\nQ:", question)
    print("  entities found by spaCy:", entities)
    show_hits(hits, limit=5)
    print(f"  -> the named recipe ({domain}) was retrieved: {has_source(hits, domain)}")
    print("  ANSWER:", reply)

**Expected:** spaCy tags the dish name (`fish and chips`, or just `chips`) as an entity. For both questions, **both** Fish and Chips recipes are retrieved at the top, so the retrieval gap from notebook 01 is gone.

**What it still cannot do.** Look at the scores: the two same-named recipes tie, with identical scores. Keyword matching finds *both*, but it cannot tell which one the question names. The website name is stored as one exact value, so the words in a question never match it. In a real run the model has to tell them apart from the context it is given.

**Try this:** call `ask_bm25` with a question that has no recipe name, such as "Is there a fish dish without sulfites?". BM25 has no sense of meaning, so a paraphrase of a recipe's wording can miss.

### Checkpoint

You should be able to say why BM25 finds both Fish and Chips recipes but cannot tell which one the question names. (The two recipes tie, because the website name is stored as one exact value that the question's words never match.)

Next: open [`03_hybrid_search.ipynb`](03_hybrid_search.ipynb).